In [24]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



In [30]:
def mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()
# Skip zero values to prevent division by zero which causes infinite/extremely high MAPE
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [5]:
# split the data
dftrain, dfdev = train_test_split(df_amazon, test_size=0.1, random_state=42)
Xtrain, ytrain, Xdev, ydev = prepare_for_train(dftrain, dfdev)


In [18]:
# Set epsilon to 1 to avoid division by very small numbers in MAPE calculation
# Keep the epsilon setting
tf.keras.backend.set_epsilon(1)

# Convert pandas Series to numpy array to normalize target variable
scaler = MinMaxScaler()
ytrain_scaled = scaler.fit_transform(ytrain.values.reshape(-1, 1)).ravel()
ydev_scaled = scaler.transform(ydev.values.reshape(-1, 1)).ravel()

# Modified model with fewer layers and different architecture
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=Xtrain.shape[1]),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)
])



In [19]:
# learning rate
initial_learning_rate = 0.0001
decay_steps = 1000
decay_rate = 0.95

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate,
    decay_steps=decay_steps,
    decay_rate=decay_rate,
    staircase=True)

# Compile with MAE loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss='mae',
    metrics=['mean_absolute_percentage_error']
)



In [20]:
# Train
history = model.fit(
    Xtrain, ytrain_scaled,
    validation_data=(Xdev, ydev_scaled),
    epochs=100,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=5,
            min_lr=1e-6
        )
    ],
    verbose=1
)

Epoch 1/100
3251/3251 [==============================] - 13s 3ms/step - loss: 0.2565 - mean_absolute_percentage_error: 25.6454 - val_loss: 0.0760 - val_mean_absolute_percentage_error: 7.6012 - lr: 8.5737e-05
Epoch 2/100
3251/3251 [==============================] - 11s 3ms/step - loss: 0.0879 - mean_absolute_percentage_error: 8.7893 - val_loss: 0.0576 - val_mean_absolute_percentage_error: 5.7626 - lr: 7.3509e-05
Epoch 3/100
3251/3251 [==============================] - 11s 3ms/step - loss: 0.0656 - mean_absolute_percentage_error: 6.5613 - val_loss: 0.0526 - val_mean_absolute_percentage_error: 5.2613 - lr: 6.3025e-05
Epoch 4/100
3251/3251 [==============================] - 11s 3ms/step - loss: 0.0579 - mean_absolute_percentage_error: 5.7942 - val_loss: 0.0513 - val_mean_absolute_percentage_error: 5.1256 - lr: 5.1334e-05
Epoch 5/100
3251/3251 [==============================] - 11s 3ms/step - loss: 0.0547 - mean_absolute_percentage_error: 5.4708 - val_loss: 0.0507 - val_mean_absolute_percen

In [ ]:
# Make predictions and calculate metrics
y_pred = model.predict(Xdev)


In [ ]:
# Calculate metrics
mae = mean_absolute_error(ydev, y_pred)
rmse = np.sqrt(mean_squared_error(ydev, y_pred))
r2 = r2_score(ydev, y_pred)

print("\nModel Results:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.2f}")

In [21]:
# Make predictions on both training and validation sets
y_pred_train = model.predict(Xtrain)
y_pred_dev = model.predict(Xdev)

367/367 [==============================] - 0s 985us/step


In [22]:
# Convert predictions back to original scale
y_pred_train = scaler.inverse_transform(y_pred_train)
y_pred_dev = scaler.inverse_transform(y_pred_dev)
ytrain_original = scaler.inverse_transform(ytrain_scaled.reshape(-1, 1))
ydev_original = scaler.inverse_transform(ydev_scaled.reshape(-1, 1))



In [31]:
# Calculate various metrics for both sets
train_mae = mean_absolute_error(ytrain_original, y_pred_train)
train_rmse = np.sqrt(mean_squared_error(ytrain_original, y_pred_train))
train_mape = mean_absolute_percentage_error(ytrain_original, y_pred_train)
train_r2 = r2_score(ytrain_original, y_pred_train)

dev_mae = mean_absolute_error(ydev_original, y_pred_dev)
dev_rmse = np.sqrt(mean_squared_error(ydev_original, y_pred_dev))
dev_mape = mean_absolute_percentage_error(ydev_original, y_pred_dev)
dev_r2 = r2_score(ydev_original, y_pred_dev)

# Print results
print("\nTraining Set Metrics:")
print(f"MAE: {train_mae:.2f}")
print(f"RMSE: {train_rmse:.2f}")
print(f"MAPE: {train_mape:.2f}%")
print(f"R2 Score: {train_r2:.4f}")

print("\nValidation Set Metrics:")
print(f"MAE: {dev_mae:.2f}")
print(f"RMSE: {dev_rmse:.2f}")
print(f"MAPE: {dev_mape:.2f}%")
print(f"R2 Score: {dev_r2:.4f}")








Training Set Metrics:
MAE: 208.10
RMSE: 277.58
MAPE: 31.97%
R2 Score: 0.0153

Validation Set Metrics:
MAE: 209.01
RMSE: 278.63
MAPE: 32.23%
R2 Score: 0.0303


In [33]:
!pip install pyparsing matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [35]:
# Display some sample predictions
print("\nSample Predictions (first 5):")
print("Actual Value | Predicted Value | Difference | % Error")
print("-" * 60)
for i in range(5):
    actual = ydev_original[i][0]
    predicted = y_pred_dev[i][0]
    diff = abs(actual - predicted)
    perc_error = (diff / actual) * 100
    print(f"${actual:10.2f} | ${predicted:10.2f} | ${diff:8.2f} | {perc_error:6.2f}%")

# Plot actual vs predicted values
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.scatter(ydev_original, y_pred_dev, alpha=0.5)
plt.plot([ydev_original.min(), ydev_original.max()], [ydev_original.min(), ydev_original.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted Values')
plt.tight_layout()
plt.show()

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss Over Time')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mean_absolute_percentage_error'], label='Training MAPE')
plt.plot(history.history['val_mean_absolute_percentage_error'], label='Validation MAPE')
plt.title('Model MAPE Over Time')
plt.xlabel('Epoch')
plt.ylabel('MAPE')
plt.legend()

plt.tight_layout()
plt.show()


Sample Predictions (first 5):
Actual Value | Predicted Value | Difference | % Error
------------------------------------------------------------
$    521.00 | $    675.79 | $  154.79 |  29.71%
$    799.00 | $    536.33 | $  262.67 |  32.87%
$   1186.00 | $    620.77 | $  565.23 |  47.66%
$    435.00 | $    473.80 | $   38.80 |   8.92%
$    744.00 | $    584.61 | $  159.39 |  21.42%


ModuleNotFoundError: No module named 'pyparsing'